In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score , f1_score , classification_report , confusion_matrix
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

In [2]:
df = pd.read_csv("shop_smart_ecommerce.csv")

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 12205 entries, 0 to 12204
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Administrative           12205 non-null  int64  
 1   Administrative_Duration  12205 non-null  float64
 2   Informational            12205 non-null  int64  
 3   Informational_Duration   12205 non-null  float64
 4   ProductRelated           12205 non-null  int64  
 5   ProductRelated_Duration  12205 non-null  float64
 6   BounceRates              12205 non-null  float64
 7   ExitRates                12205 non-null  float64
 8   PageValues               12205 non-null  float64
 9   SpecialDay               12205 non-null  float64
 10  Month                    12205 non-null  str    
 11  OperatingSystems         12205 non-null  int64  
 12  Browser                  12205 non-null  int64  
 13  Region                   12205 non-null  int64  
 14  TrafficType              12205 no

In [3]:
df = df.drop_duplicates().reset_index(drop=True)

In [4]:
x = df.drop(columns = ["Revenue"])
y = df["Revenue"].astype(int)

In [5]:
num_cols = x.select_dtypes(include=['int64', 'float64']).columns
cat_cols = x.select_dtypes(include=['object', 'str']).columns

In [6]:
x_train , x_test , y_train , y_test = train_test_split(
    x , y , test_size = 0.2 , random_state = 42 , stratify=y
)

In [7]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols),
        
    ],
    remainder = "passthrough"
)

In [8]:
from sklearn.tree import DecisionTreeClassifier

dt = DecisionTreeClassifier(
    max_depth = 6,              # prevents deep overfitting
    min_samples_leaf = 30,      # smooths decision boundaries
    class_weight = "balanced",  # handles imbalance
    random_state = 42 
)

In [9]:
model = Pipeline([
    ("prep", preprocessor),
    ("dt", dt),
])

model.fit(x_train, y_train)

train_pred = model.predict(x_train)
test_pred = model.predict(x_test)

print("Depth:", model.named_steps["dt"].get_depth(),
      "| Leaves:", model.named_steps["dt"].get_n_leaves())
print("Train F1:", round(f1_score(y_train, train_pred), 3))
print("Test  F1:", round(f1_score(y_test, test_pred), 3))
print()
print(classification_report(y_test, test_pred))
print(confusion_matrix(y_test, test_pred))

Depth: 6 | Leaves: 52
Train F1: 0.653
Test  F1: 0.625

              precision    recall  f1-score   support

           0       0.96      0.85      0.90      2059
           1       0.50      0.83      0.63       382

    accuracy                           0.84      2441
   macro avg       0.73      0.84      0.76      2441
weighted avg       0.89      0.84      0.86      2441

[[1742  317]
 [  64  318]]


In [10]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "dt__max_depth": [3, 4, 5, 6, 8, 10],
    "dt__min_samples_split": [10, 30, 50, 100],
    "dt__min_samples_leaf": [5, 10, 20, 30, 50],
    "dt__class_weight": [None, "balanced"],
}

grid = GridSearchCV(model, param_grid, scoring="f1", cv=5, n_jobs=-1)
grid.fit(x_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV F1:", round(grid.best_score_, 3))

best = grid.best_estimator_
print("Train F1:", round(f1_score(y_train, best.predict(x_train)), 3))
print("Test  F1:", round(f1_score(y_test, best.predict(x_test)), 3))

Best params: {'dt__class_weight': None, 'dt__max_depth': 6, 'dt__min_samples_leaf': 50, 'dt__min_samples_split': 10}
Best CV F1: 0.656
Train F1: 0.673
Test  F1: 0.649
